In [1]:
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv
import os

load_dotenv()
print('Librerías cargadas')

Librerías cargadas


In [2]:
ARCHIVO = 'datos/Base Haciendas Depurada.txt'

df = pd.read_csv(
    ARCHIVO,
    sep='\t',
    encoding='utf-16',
    decimal=',',
    thousands='.'
)

df['FECHA'] = pd.to_datetime(df['FECHA'], format='%d/%m/%Y').dt.date

print(f'Filas: {len(df):,}  |  Columnas: {len(df.columns)}')
df.head(3)

Filas: 2,904  |  Columnas: 53


,FECHA,Semana,Zona,Unidad,Nombre_Unidad,Real,Costo_Ha,Atencion_Plantacion,C_Riego,C_Mano_Obra_Riego,...,Precipitacion_mm,Evotranspiracion,Humedad,Ausentismo_Agricola,Ausentismo_Justificado_Agricola,Ausentismo_Injustificado_Agricola,RotPerson_Salida_Todos_Motivos_Agricola,Pago_Labor_Persona,Pago_Por_Cuenta,Vacante_Labor
0,2020-01-01,5,Zona Camarones,2283,Hacienda San Escriva,5.114322,926.408722,120674.0,1576,643,...,84.40,14.00,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0
1,2020-02-01,9,Zona Camarones,2283,Hacienda San Escriva,6.033120,1010.523197,131428.0,2253,540,...,112.25,17.00,0.0,0.0,0.0,0.0,0.006803,0.0,0.0,0
2,2020-03-01,14,Zona Camarones,2283,Hacienda San Escriva,4.495248,831.462938,107817.0,778,515,...,101.75,16.25,0.0,0.0,0.0,0.0,0.013423,0.0,0.0,0


In [3]:
# Ver tipos de datos inferidos
df.dtypes

FECHA                                       object
Semana                                       int64
Zona                                           str
Unidad                                       int64
Nombre_Unidad                                  str
Real                                       float64
Costo_Ha                                   float64
Atencion_Plantacion                        float64
C_Riego                                      int64
C_Mano_Obra_Riego                            int64
C_Mantenimiento_Riego                        int64
C_Combustible                                int64
C_Control_Sigatoca                           int64
C_Aplicacion_Aerea                           int64
C_Deshoje                                    int64
C_Costos_Productos                           int64
C_Fertilizacion                              int64
C_Sacos_Fert                                 int64
C_ManodeObra_Fert                            int64
C_Transporte_Fert              

In [4]:
TABLA = 'haciendas'

conn = psycopg2.connect(
    user=os.getenv('user'),
    password=os.getenv('password'),
    host=os.getenv('host'),
    port=os.getenv('port'),
    dbname=os.getenv('dbname')
)

import datetime

def pg_type(col):
    if col == 'FECHA':
        return 'DATE'
    dtype = df[col].dtype
    if pd.api.types.is_integer_dtype(dtype):
        return 'BIGINT'
    elif pd.api.types.is_float_dtype(dtype):
        return 'DOUBLE PRECISION'
    else:
        return 'TEXT'

cols_ddl = ',\n  '.join(f'"{c}" {pg_type(c)}' for c in df.columns)
ddl = f'DROP TABLE IF EXISTS {TABLA};\nCREATE TABLE {TABLA} (\n  {cols_ddl}\n);'

print(ddl)

DROP TABLE IF EXISTS haciendas;
CREATE TABLE haciendas (
  "FECHA" DATE,
  "Semana" BIGINT,
  "Zona" TEXT,
  "Unidad" BIGINT,
  "Nombre_Unidad" TEXT,
  "Real" DOUBLE PRECISION,
  "Costo_Ha" DOUBLE PRECISION,
  "Atencion_Plantacion" DOUBLE PRECISION,
  "C_Riego" BIGINT,
  "C_Mano_Obra_Riego" BIGINT,
  "C_Mantenimiento_Riego" BIGINT,
  "C_Combustible" BIGINT,
  "C_Control_Sigatoca" BIGINT,
  "C_Aplicacion_Aerea" BIGINT,
  "C_Deshoje" BIGINT,
  "C_Costos_Productos" BIGINT,
  "C_Fertilizacion" BIGINT,
  "C_Sacos_Fert" BIGINT,
  "C_ManodeObra_Fert" BIGINT,
  "C_Transporte_Fert" BIGINT,
  "C_Administracion_Hacienda" BIGINT,
  "Sueldos" BIGINT,
  "Servicios_Basicos" BIGINT,
  "C_Empaque_Fijo" BIGINT,
  "Mantenimiento_Empacadora" BIGINT,
  "Mantenimiento_Equipo" BIGINT,
  "C_Logistica" DOUBLE PRECISION,
  "Transporte" BIGINT,
  "Materiales" BIGINT,
  "Reclasificaciones_Transporte" BIGINT,
  "Reclasificaciones_Materiales" BIGINT,
  "C_Empaque_Variable" BIGINT,
  "C_Cosecha" BIGINT,
  "C_Transpo

In [5]:
# Crear la tabla e insertar datos
cur = conn.cursor()
cur.execute(ddl)

cols = [f'"{c}"' for c in df.columns]
query = f'INSERT INTO {TABLA} ({(", ").join(cols)}) VALUES %s'

# Reemplazar NaN por None (NULL en Postgres)
rows = [tuple(None if pd.isna(v) else v for v in row) for row in df.itertuples(index=False)]

execute_values(cur, query, rows, page_size=500)
conn.commit()
cur.close()
conn.close()

print(f'✓ {len(rows):,} filas insertadas en la tabla "{TABLA}"')

✓ 2,904 filas insertadas en la tabla "haciendas"
